
# GOLD91 BAGPIPES SFH-model robustness test

This notebook tests whether the inferred backward luminosity factor at \(z=3\),

\[
g_{\rm BAG} =
\frac{L_{\rm rest}(z=3)}
     {L_{\rm rest}(z_{\rm source})},
\]

is robust to the assumed star-formation history (SFH).

It deliberately changes **only the SFH model** while keeping fixed the photometric setup that worked best in the pilot:

- objects: 487469 and 756229;
- fixed COSMOS2025 catalog redshift;
- SE++ model photometry;
- calibrated model-flux uncertainties;
- maximum S/N = 20, equivalent to a 5% minimum fractional uncertainty;
- IRAC ch1/ch2 excluded;
- same broad-band filter set;
- Calzetti dust;
- no nebular component;
- same metallicity and formed-mass prior philosophy.

The three SFH models are:

1. double-power-law (DPL), identical to the accepted no-IRAC pilot;
2. delayed-\(\tau\);
3. Leja-style non-parametric continuity SFH.

The existing DPL result is **reused automatically** if its BAGPIPES posterior files are already present. BAGPIPES does not refit an existing run.

Run this notebook with **Python (bagpipes)**.


In [ ]:

from pathlib import Path
from copy import deepcopy
from io import BytesIO
from urllib.parse import quote
from urllib.request import urlopen
import os, sys, json, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.io.votable import parse_single_table

import bagpipes as pipes

print("Python:", sys.executable)
print("BAGPIPES:", pipes.__file__)

if "/envs/bagpipes/" not in sys.executable.replace("\\", "/"):
    print("\nWARNING: select the 'Python (bagpipes)' kernel in VS Code.")


In [ ]:

# ------------------------------------------------------------------
# PATHS AND CONFIGURATION
# ------------------------------------------------------------------

PROJECT_ROOT = Path("/home/bahareh/Desktop/Projects/Passive_Spiral")

MASTER_CATALOG = Path(
    "/home/bahareh/Desktop/Projects/Data_General/Cosmos_Web/"
    "COSMOSWeb_mastercatalog_v1.1.fits"
)

PRODUCTION_DIR = (
    PROJECT_ROOT
    / "Data/forward_z3_GOLD403_v1/model_based_v1/production_frozen_20260919T2115"
)

GOLD91_CSV = (
    PRODUCTION_DIR
    / "posthoc_E0_audit/BAGPIPES_GOLD91_factor_request.csv"
)

# Reuse the same work directory as the successful pilot so that the
# already-computed DPL posteriors can be loaded instead of refitted.
WORKDIR = PROJECT_ROOT / "Data/bagpipes_GOLD91_pilot_v1"
FILTER_DIR = WORKDIR / "filters"
REST_FILTER_DIR = WORKDIR / "matched_rest_filters"
RESULT_DIR = WORKDIR / "sfh_model_robustness"

for d in [
    WORKDIR,
    FILTER_DIR,
    REST_FILTER_DIR,
    RESULT_DIR,
    WORKDIR / "pipes",
    WORKDIR / "pipes/posterior",
    WORKDIR / "pipes/plots",
    WORKDIR / "pipes/cats",
]:
    d.mkdir(parents=True, exist_ok=True)

os.chdir(WORKDIR)

TEST_IDS = [487469, 756229]
Z_TARGET = 3.0

# Same no-IRAC band set used in the accepted pilot.
FILTER_INFO = [
    ("hsc-g",     "Subaru/HSC.g"),
    ("hsc-r",     "Subaru/HSC.r"),
    ("hsc-i",     "Subaru/HSC.i"),
    ("hsc-z",     "Subaru/HSC.z"),
    ("hsc-y",     "Subaru/HSC.Y"),
    ("uvista-y",  "Paranal/VISTA.Y"),
    ("uvista-j",  "Paranal/VISTA.J"),
    ("uvista-h",  "Paranal/VISTA.H"),
    ("uvista-ks", "Paranal/VISTA.Ks"),
    ("f115w",     "JWST/NIRCam.F115W"),
    ("f150w",     "JWST/NIRCam.F150W"),
    ("f277w",     "JWST/NIRCam.F277W"),
    ("f444w",     "JWST/NIRCam.F444W"),
]

N_POSTERIOR = 400
N_FACTOR_DRAWS = 160

RUN_NAMES = {
    # EXACT same run name as the successful no-IRAC DPL test.
    "dblplaw":   "gold91_dblplaw_pilot_snrfloor_noirac_v1",
    "delayed":   "gold91_delayed_pilot_snrfloor_noirac_v1",
    "continuity":"gold91_continuity_pilot_snrfloor_noirac_v1",
}

print("WORKDIR:", WORKDIR)
print("RESULT_DIR:", RESULT_DIR)



## 1. Load the two test galaxies

These are the two clean pilot galaxies for which COSMOS2025 LePHARE fits were reasonable. The problematic third pilot object is intentionally excluded from this SFH-model robustness test.


In [ ]:

gold91 = pd.read_csv(GOLD91_CSV)
gold91["source_id"] = pd.to_numeric(gold91["source_id"], errors="raise").astype(int)

if len(gold91) != 91:
    raise RuntimeError(f"Expected 91 rows in GOLD91 table, found {len(gold91)}.")

pilot = (
    gold91.loc[gold91["source_id"].isin(TEST_IDS)]
    .set_index("source_id")
    .loc[TEST_IDS]
    .reset_index()
)

if len(pilot) != 2:
    raise RuntimeError("One or both test IDs are missing from the GOLD91 table.")

display(
    pilot[
        ["source_id", "catalog_z", "logM", "mag_model_f444w", "g_E1J", "dmag_E1J"]
    ]
)



## 2. Load/cache the exact same no-IRAC filter set

Existing filter files from the previous pilot are reused. Missing files are downloaded from the SVO Filter Profile Service.


In [ ]:

def download_svo_filter(filter_id, outfile, timeout=90):
    outfile = Path(outfile)

    if outfile.is_file() and outfile.stat().st_size > 100:
        arr = np.loadtxt(outfile)
        if arr.ndim == 2 and arr.shape[1] >= 2 and len(arr) >= 5:
            return outfile

    url = (
        "https://svo2.cab.inta-csic.es/theory/fps/fps.php?ID="
        + quote(filter_id, safe="/")
    )

    print("Downloading", filter_id)

    with urlopen(url, timeout=timeout) as response:
        raw = response.read()

    table = parse_single_table(BytesIO(raw)).to_table(use_names_over_ids=True)
    names = {str(c).lower(): str(c) for c in table.colnames}

    if "wavelength" not in names or "transmission" not in names:
        raise RuntimeError(
            f"SVO response for {filter_id} lacks Wavelength/Transmission columns."
        )

    wav = np.asarray(table[names["wavelength"]], dtype=float)
    trans = np.asarray(table[names["transmission"]], dtype=float)

    good = np.isfinite(wav) & np.isfinite(trans)
    wav, trans = wav[good], trans[good]

    order = np.argsort(wav)
    wav, trans = wav[order], trans[order]

    if len(wav) < 5 or np.nanmax(trans) <= 0:
        raise RuntimeError(f"Invalid SVO transmission curve: {filter_id}")

    trans = trans / np.nanmax(trans)
    np.savetxt(outfile, np.c_[wav, trans], fmt="%.8e")

    return outfile


FILTER_SUFFIXES = []
FILTER_PATHS = []

for suffix, svo_id in FILTER_INFO:
    path = FILTER_DIR / f"{suffix}.dat"
    download_svo_filter(svo_id, path)
    FILTER_SUFFIXES.append(suffix)
    FILTER_PATHS.append(str(path))

print("N filters:", len(FILTER_PATHS))
print(FILTER_SUFFIXES)



## 3. Extract COSMOS2025 photometry and apply the accepted S/N floor

The likelihood input is kept identical across the three SFH models.

For every valid retained band,

\[
\sigma_{\rm used} =
\max\left(\sigma_{\rm catalog},\frac{|F|}{20}\right).
\]

Thus the fit never assigns S/N above 20. Negative flux measurements are retained.


In [ ]:

def first_table_hdu(hdul):
    for i, hdu in enumerate(hdul):
        if isinstance(hdu, fits.BinTableHDU):
            return i
    raise RuntimeError("No binary-table HDU found.")


raw_phot_by_id = {}
phot_by_id = {}

with fits.open(MASTER_CATALOG, memmap=True) as hdul:
    ihdu = first_table_hdu(hdul)
    data = hdul[ihdu].data
    names = list(data.names)

    id_col = next(
        (c for c in ["id", "ID", "source_id", "SOURCE_ID"] if c in names),
        None
    )
    if id_col is None:
        raise KeyError("Could not identify source-ID column.")

    required_cols = []
    for suffix in FILTER_SUFFIXES:
        required_cols += [
            f"flux_model_{suffix}",
            f"flux_err-cal_model_{suffix}",
        ]

    absent = [c for c in required_cols if c not in names]
    if absent:
        raise KeyError("Missing catalog columns:\n" + "\n".join(absent))

    all_ids = np.asarray(data[id_col])

    for oid in TEST_IDS:
        hit = np.flatnonzero(all_ids == oid)

        if len(hit) != 1:
            raise RuntimeError(
                f"source_id={oid}: expected one row, found {len(hit)}."
            )

        row = data[int(hit[0])]

        raw = []

        for suffix in FILTER_SUFFIXES:
            f = float(row[f"flux_model_{suffix}"])
            e = float(row[f"flux_err-cal_model_{suffix}"])

            valid = (
                np.isfinite(f)
                and np.isfinite(e)
                and e > 0
                and abs(f) < 1e20
                and e < 1e20
            )

            if valid:
                raw.append([f, e])
            else:
                raw.append([0.0, 9.9e99])

        raw = np.asarray(raw, dtype=float)
        raw_phot_by_id[oid] = raw.copy()

        floored = raw.copy()

        for j in range(len(floored)):
            f, e = floored[j]

            if np.isfinite(e) and 0 < e < 1e50:
                floored[j, 1] = max(e, abs(f) / 20.0)

        phot_by_id[oid] = floored


def load_photometry(ID):
    return np.asarray(phot_by_id[int(ID)], dtype=float)


# Quick audit.
audit_rows = []
for oid in TEST_IDS:
    for band, (f0, e0), (_, e1) in zip(
        FILTER_SUFFIXES,
        raw_phot_by_id[oid],
        phot_by_id[oid],
    ):
        audit_rows.append({
            "source_id": oid,
            "band": band,
            "flux_uJy": f0,
            "catalog_err_uJy": e0 if e0 < 1e50 else np.nan,
            "used_err_uJy": e1 if e1 < 1e50 else np.nan,
            "catalog_snr": f0/e0 if e0 < 1e50 else np.nan,
            "used_snr": f0/e1 if e1 < 1e50 else np.nan,
        })

phot_audit = pd.DataFrame(audit_rows)
display(phot_audit)



## 4. Build the three SFH model families

The dust prescription and mass/metallicity prior philosophy are kept fixed.

### DPL

This reproduces the existing no-IRAC pilot model.

### Delayed-\(\tau\)

\[
{\rm SFR}(t)\propto t\,e^{-t/\tau}.
\]

The onset age and \(\tau\) are fitted.

### Continuity

A non-parametric piecewise SFH with Student-t priors on changes in adjacent SFR bins. One bin edge is deliberately placed at the lookback time from the observed galaxy to \(z=3\), so the model can separately represent stars formed before and after the target epoch instead of burying \(z=3\) inside one very broad ancient bin.


In [ ]:

def universe_age_gyr(z):
    return float(np.interp(float(z), pipes.utils.z_array, pipes.utils.age_at_z))


def common_mass_prior(logm_guess):
    return (
        max(7.0, float(logm_guess) - 1.5),
        min(13.5, float(logm_guess) + 1.5),
    )


def make_dpl_instructions(z, logm_guess):
    age_univ = universe_age_gyr(z)

    return {
        "redshift": float(z),

        "dblplaw": {
            "tau": (0.1, min(13.0, 1.45 * age_univ)),
            "alpha": (0.01, 1000.0),
            "alpha_prior": "log_10",
            "beta": (0.01, 1000.0),
            "beta_prior": "log_10",
            "massformed": common_mass_prior(logm_guess),
            "metallicity": (0.05, 2.5),
            "metallicity_prior": "log_10",
        },

        "dust": {
            "type": "Calzetti",
            "Av": (0.0, 2.0),
        },
    }


def make_delayed_instructions(z, logm_guess):
    age_univ = universe_age_gyr(z)

    return {
        "redshift": float(z),

        "delayed": {
            # Time since star formation began.
            "age": (0.10, 0.999 * age_univ),

            # Broad timescale prior. Log prior avoids over-weighting long tau.
            "tau": (0.03, 10.0),
            "tau_prior": "log_10",

            "massformed": common_mass_prior(logm_guess),
            "metallicity": (0.05, 2.5),
            "metallicity_prior": "log_10",
        },

        "dust": {
            "type": "Calzetti",
            "Av": (0.0, 2.0),
        },
    }


def make_continuity_instructions(z, logm_guess, z_target=Z_TARGET):
    age_univ = universe_age_gyr(z)
    age_target = universe_age_gyr(z_target)

    # Lookback time, from observation epoch, to z=3.
    delta_z3_myr = 1000.0 * (age_univ - age_target)

    # Last edge safely inside the age of the Universe.
    max_age_myr = 1000.0 * age_univ - 1.0

    # Standard fine recent bins plus an explicit z=3 boundary.
    candidate_edges = [
        0.0,
        30.0,
        100.0,
        300.0,
        1000.0,
        3000.0,
        delta_z3_myr,
        max_age_myr,
    ]

    # Sort, remove accidental near-duplicates, and guarantee strict increase.
    edges = []
    for x in sorted(candidate_edges):
        if x < 0 or x > max_age_myr:
            continue
        if not edges or (x - edges[-1]) > 5.0:
            edges.append(float(x))

    if edges[0] != 0.0:
        edges.insert(0, 0.0)

    if (max_age_myr - edges[-1]) > 5.0:
        edges.append(float(max_age_myr))

    continuity = {
        "massformed": common_mass_prior(logm_guess),
        "metallicity": (0.05, 2.5),
        "metallicity_prior": "log_10",
        "bin_edges": edges,
    }

    # One dsfr parameter between every pair of adjacent bins.
    for i in range(1, len(edges) - 1):
        continuity[f"dsfr{i}"] = (-10.0, 10.0)
        continuity[f"dsfr{i}_prior"] = "student_t"
        # BAGPIPES defaults correspond to the Leja+ continuity prior:
        # scale=0.3, df=2. We leave those defaults unchanged.

    return {
        "redshift": float(z),
        "continuity": continuity,

        "dust": {
            "type": "Calzetti",
            "Av": (0.0, 2.0),
        },
    }


FIT_BUILDERS = {
    "dblplaw": make_dpl_instructions,
    "delayed": make_delayed_instructions,
    "continuity": make_continuity_instructions,
}


fit_instructions = {
    model: {}
    for model in FIT_BUILDERS
}

for oid in TEST_IDS:
    row = pilot.loc[pilot["source_id"] == oid].iloc[0]
    z = float(row["catalog_z"])
    logm = float(row["logM"])

    for model, builder in FIT_BUILDERS.items():
        fit_instructions[model][oid] = builder(z, logm)


# Show continuity bin edges because the z=3 edge is scientifically important.
for oid in TEST_IDS:
    print(
        "\nID", oid,
        "continuity bin edges [Myr lookback from source epoch]:",
        fit_instructions["continuity"][oid]["continuity"]["bin_edges"]
    )



## 5. Create BAGPIPES galaxy objects

The same photometry is passed to every SFH family.


In [ ]:

galaxies = {}

for oid in TEST_IDS:
    galaxies[oid] = pipes.galaxy(
        str(oid),
        load_photometry,
        spectrum_exists=False,
        filt_list=FILTER_PATHS,
        phot_units="mujy",
    )

print("Galaxy objects ready:", sorted(galaxies))



## 6. Fit DPL, delayed-\(\tau\), and continuity

The DPL run name is exactly the same as the successful previous pilot. If those `.h5` files exist, BAGPIPES prints `Results loaded ...` and skips fitting them.

Delayed-\(\tau\) and continuity use new run names.

The continuity model has more parameters, so it uses more Nautilus live points.


In [ ]:

fits = {model: {} for model in FIT_BUILDERS}

for model in ["dblplaw", "delayed", "continuity"]:

    print("\n" + "#" * 78)
    print("MODEL:", model)
    print("#" * 78)

    for oid in TEST_IDS:

        print("\n" + "=" * 70)
        print("OBJECT:", oid)
        print("=" * 70)

        fit = pipes.fit(
            galaxies[oid],
            fit_instructions[model][oid],
            run=RUN_NAMES[model],
            n_posterior=N_POSTERIOR,
        )

        # Existing posteriors are automatically loaded and not refitted.
        n_live = 400 if model == "continuity" else 250

        fit.fit(
            verbose=False,
            sampler="nautilus",
            n_live=n_live,
        )

        fits[model][oid] = fit

print("\nAll requested model/object combinations are available.")



## 7. Compare observed-epoch fit quality

We report best \(\chi^2\), \(\chi^2/{\rm dof}\), AIC, BIC, and BAGPIPES log evidence.

Because the three models do not have equal dimensionality, raw reduced \(\chi^2\) alone is not enough. AIC/BIC provide simple complexity-penalized checks. The nested-sampling log evidence is also recorded, but the luminosity-factor robustness test below remains the main goal.


In [ ]:

fit_quality_rows = []

n_valid = int(
    np.sum(
        np.isfinite(phot_by_id[TEST_IDS[0]][:, 1])
        & (phot_by_id[TEST_IDS[0]][:, 1] > 0)
        & (phot_by_id[TEST_IDS[0]][:, 1] < 1e50)
    )
)

for model, model_fits in fits.items():

    for oid, fit in model_fits.items():

        post = fit.posterior

        if "chisq_phot" not in post.samples:
            post.get_advanced_quantities()

        chisq = np.asarray(post.samples["chisq_phot"], dtype=float)

        k = int(fit.fitted_model.ndim)
        dof = max(n_valid - k, 1)
        chi2_best = float(np.nanmin(chisq))

        lnz = np.nan
        lnz_err = np.nan

        if "lnz" in fit.results:
            lnz = float(np.asarray(fit.results["lnz"]).squeeze())

        if "lnz_err" in fit.results:
            lnz_err = float(np.asarray(fit.results["lnz_err"]).squeeze())

        fit_quality_rows.append({
            "source_id": oid,
            "sfh_model": model,
            "n_bands": n_valid,
            "n_parameters": k,
            "dof": dof,
            "chi2_best": chi2_best,
            "chi2_median_posterior": float(np.nanmedian(chisq)),
            "reduced_chi2_best": chi2_best / dof,
            "AIC": chi2_best + 2.0 * k,
            "BIC": chi2_best + k * np.log(n_valid),
            "lnZ": lnz,
            "lnZ_err": lnz_err,
        })

fit_quality = pd.DataFrame(fit_quality_rows).sort_values(
    ["source_id", "sfh_model"]
).reset_index(drop=True)

display(fit_quality)

fit_quality.to_csv(
    RESULT_DIR / "SFH_model_fit_quality.csv",
    index=False,
)



## 8. Save SED and SFH posterior plots

This produces one SED and one SFH posterior plot per object/model combination under the normal BAGPIPES plot directories.


In [ ]:

for model, model_fits in fits.items():

    for oid, fit in model_fits.items():

        print("\nPlotting", model, oid)

        fit.plot_spectrum_posterior(
            save=True,
            show=False,
        )

        fit.plot_sfh_posterior(
            save=True,
            show=False,
        )

print("Plots saved.")



## 9. Define the common rest-frame F444W-at-\(z=3\) band

We compare source-epoch and \(z=3\) populations in exactly the same intrinsic rest-frame band.

The F444W transmission wavelength axis is divided by \(1+3\), then both epochs are evaluated at BAGPIPES `redshift=0`. This removes luminosity distance, observer-frame projection, and IGM from \(g_{\rm BAG}\).


In [ ]:

F444_PATH = Path(
    FILTER_PATHS[FILTER_SUFFIXES.index("f444w")]
)

arr = np.loadtxt(F444_PATH)
wav_obs, trans = arr[:, 0], arr[:, 1]

REST_FILTER_PATH = (
    REST_FILTER_DIR
    / "F444W_rest_at_z3.dat"
)

np.savetxt(
    REST_FILTER_PATH,
    np.c_[wav_obs / (1.0 + Z_TARGET), trans],
    fmt="%.8e",
)

print("Rest filter:", REST_FILTER_PATH)



## 10. General SFH truncation and \(g_{\rm BAG}\) calculation

This implementation is model-agnostic: it works with DPL, delayed-\(\tau\), and continuity because it reconstructs BAGPIPES' posterior SFH array first, then explicitly removes stars that had not formed by \(z=3\).

Dust is not applied to the intrinsic luminosity ratio. Dust is still fitted at the observed epoch because it affects the inferred SFH.


In [ ]:

SFH_COMPONENT = {
    "dblplaw": "dblplaw",
    "delayed": "delayed",
    "continuity": "continuity",
}


def make_custom_components_from_posterior_sfh(
    source_components,
    source_sfh_obj,
    sfh_component_name,
    z_source,
    z_target=Z_TARGET,
):
    """
    Build dust-free custom-SFH components for source and target epochs.
    """

    z_source = float(z_source)
    z_target = float(z_target)

    source_age_univ_yr = (
        universe_age_gyr(z_source) * 1e9
    )

    target_age_univ_yr = (
        universe_age_gyr(z_target) * 1e9
    )

    if target_age_univ_yr >= source_age_univ_yr:
        raise ValueError("Target epoch must be earlier than source epoch.")

    ages_src = np.asarray(source_sfh_obj.ages, dtype=float)
    sfr_src = np.asarray(source_sfh_obj.sfh, dtype=float)

    comp = source_components[sfh_component_name]

    logm_now = float(comp["massformed"])
    metallicity = float(comp["metallicity"])

    with np.errstate(divide="ignore", invalid="ignore"):
        logm_target = float(
            source_sfh_obj.massformed_at_redshift(z_target)
        )

    if not np.isfinite(logm_target):
        return None, None, 0.0, np.nan

    formed_fraction = 10.0 ** (logm_target - logm_now)

    source_mask = (
        np.isfinite(ages_src)
        & np.isfinite(sfr_src)
        & (ages_src >= 0)
        & (ages_src <= source_age_univ_yr)
    )

    ages_src_use = ages_src[source_mask]
    sfr_src_use = sfr_src[source_mask]

    # A star of age A at z_source had age A-delta_t at z_target.
    delta_t = source_age_univ_yr - target_age_univ_yr

    ages_target = ages_src_use - delta_t
    target_mask = ages_target >= 0

    ages_target_use = ages_target[target_mask]
    sfr_target_use = sfr_src_use[target_mask]

    if (
        len(ages_src_use) < 3
        or len(ages_target_use) < 3
        or not np.any(sfr_target_use > 0)
    ):
        return None, None, formed_fraction, logm_target

    source_history = np.c_[
        np.r_[0.0, ages_src_use],
        np.r_[sfr_src_use[0], sfr_src_use],
    ]

    target_history = np.c_[
        np.r_[0.0, ages_target_use],
        np.r_[sfr_target_use[0], sfr_target_use],
    ]

    src_custom = {
        "redshift": z_source,
        "custom": {
            "history": source_history,
            "massformed": logm_now,
            "metallicity": metallicity,
        },
    }

    tgt_custom = {
        "redshift": z_target,
        "custom": {
            "history": target_history,
            "massformed": logm_target,
            "metallicity": metallicity,
        },
    }

    return (
        src_custom,
        tgt_custom,
        formed_fraction,
        logm_target,
    )


def intrinsic_rest_photometry(
    custom_components,
    rest_filter_path=REST_FILTER_PATH,
):
    """
    Intrinsic dust-free luminosity density through one rest-frame filter.

    redshift=0 means BAGPIPES does not divide by luminosity distance.
    Default photometry units avoid the current single-filter muJy
    broadcasting edge case.
    """

    comp = deepcopy(custom_components)
    comp["redshift"] = 0.0

    model = pipes.model_galaxy(
        comp,
        filt_list=[str(rest_filter_path)],
    )

    return float(
        np.asarray(model.photometry).reshape(-1)[0]
    )


def bag_factor_draws_for_model(
    fit,
    oid,
    sfh_model,
    n_draws=N_FACTOR_DRAWS,
    seed=260926,
):
    z_source = float(
        pilot.loc[
            pilot["source_id"] == oid,
            "catalog_z",
        ].iloc[0]
    )

    post = fit.posterior
    n_all = post.samples2d.shape[0]

    rng = np.random.default_rng(
        seed
        + int(oid)
        + {"dblplaw": 1, "delayed": 2, "continuity": 3}[sfh_model]
    )

    take = min(int(n_draws), n_all)

    inds = rng.choice(
        n_all,
        size=take,
        replace=False,
    )

    rows = []

    for ind in inds:

        params = post.samples2d[ind, :]

        fit.fitted_model._update_model_components(params)
        components = deepcopy(
            fit.fitted_model.model_components
        )

        # Rebuild the SFH associated with this posterior draw.
        post.sfh.update(components)

        (
            src_custom,
            tgt_custom,
            formed_fraction,
            logm_z3,
        ) = make_custom_components_from_posterior_sfh(
            components,
            post.sfh,
            SFH_COMPONENT[sfh_model],
            z_source,
            Z_TARGET,
        )

        if src_custom is None or tgt_custom is None:
            rows.append({
                "source_id": oid,
                "sfh_model": sfh_model,
                "posterior_index": int(ind),
                "valid": False,
                "reason": "NO_STELLAR_HISTORY_AT_Z3",
                "formed_mass_fraction_z3": formed_fraction,
                "logMformed_z3": logm_z3,
                "g_bag": np.nan,
            })
            continue

        try:
            L_source = intrinsic_rest_photometry(
                src_custom
            )

            L_z3 = intrinsic_rest_photometry(
                tgt_custom
            )

            if not (
                np.isfinite(L_source)
                and np.isfinite(L_z3)
                and L_source > 0
                and L_z3 >= 0
            ):
                raise ValueError(
                    "NONFINITE_OR_NONPOSITIVE_INTRINSIC_LUMINOSITY"
                )

            g = L_z3 / L_source

            rows.append({
                "source_id": oid,
                "sfh_model": sfh_model,
                "posterior_index": int(ind),
                "valid": np.isfinite(g) and g >= 0,
                "reason": "",
                "Lrest_source": L_source,
                "Lrest_z3": L_z3,
                "g_bag": g,
                "formed_mass_fraction_z3": formed_fraction,
                "logMformed_z3": logm_z3,
            })

        except Exception as exc:
            rows.append({
                "source_id": oid,
                "sfh_model": sfh_model,
                "posterior_index": int(ind),
                "valid": False,
                "reason": repr(exc),
                "formed_mass_fraction_z3": formed_fraction,
                "logMformed_z3": logm_z3,
                "g_bag": np.nan,
            })

    return pd.DataFrame(rows)


In [ ]:

all_factor_draws = []

for model, model_fits in fits.items():

    for oid, fit in model_fits.items():

        print(
            "Computing g_BAG:",
            "model =", model,
            "ID =", oid,
        )

        draws = bag_factor_draws_for_model(
            fit,
            oid,
            model,
        )

        all_factor_draws.append(draws)


factor_draws = pd.concat(
    all_factor_draws,
    ignore_index=True,
)

factor_draws.to_csv(
    RESULT_DIR / "SFH_model_gbag_draws.csv",
    index=False,
)

display(
    factor_draws.groupby(
        ["source_id", "sfh_model"]
    )["valid"].mean().rename("valid_fraction")
)



## 11. Summarize \(g_{\rm BAG}\) and the assembled stellar mass at \(z=3\)

Besides the 16th/50th/84th percentiles, the table reports:

- probability \(g_{\rm BAG}>1\);
- probability \(g_{\rm BAG}>0.5\);
- probability that at least 10% of the final formed stellar mass already existed by \(z=3\);
- fraction of posterior draws with no stellar history at \(z=3\);
- width of the 16-84% interval in dex where defined.


In [ ]:

def summarize_group(group):

    g = pd.to_numeric(
        group["g_bag"],
        errors="coerce",
    )

    valid = (
        group["valid"].fillna(False).astype(bool)
        & np.isfinite(g)
        & (g >= 0)
    )

    good = group.loc[valid].copy()

    no_prog = (
        group["reason"]
        .fillna("")
        .eq("NO_STELLAR_HISTORY_AT_Z3")
        .mean()
    )

    if len(good) == 0:
        return pd.Series({
            "n_draws": len(group),
            "n_valid": 0,
            "valid_fraction": 0.0,
            "g_p16": np.nan,
            "g_p50": np.nan,
            "g_p84": np.nan,
            "g_span_dex_16_84": np.nan,
            "P_g_gt_1": np.nan,
            "P_g_gt_0p5": np.nan,
            "formed_fraction_p16": np.nan,
            "formed_fraction_p50": np.nan,
            "formed_fraction_p84": np.nan,
            "P_formed_fraction_gt_0p1": np.nan,
            "no_z3_progenitor_fraction": no_prog,
        })

    gvals = np.asarray(good["g_bag"], dtype=float)
    fvals = np.asarray(
        good["formed_mass_fraction_z3"],
        dtype=float,
    )

    gp = np.percentile(gvals, [16, 50, 84])
    fp = np.percentile(fvals, [16, 50, 84])

    span_dex = np.nan

    if gp[0] > 0 and gp[2] > 0:
        span_dex = np.log10(gp[2]) - np.log10(gp[0])

    return pd.Series({
        "n_draws": len(group),
        "n_valid": len(good),
        "valid_fraction": len(good) / len(group),

        "g_p16": gp[0],
        "g_p50": gp[1],
        "g_p84": gp[2],
        "g_span_dex_16_84": span_dex,

        "P_g_gt_1": np.mean(gvals > 1.0),
        "P_g_gt_0p5": np.mean(gvals > 0.5),

        "formed_fraction_p16": fp[0],
        "formed_fraction_p50": fp[1],
        "formed_fraction_p84": fp[2],

        "P_formed_fraction_gt_0p1": np.mean(
            fvals > 0.1
        ),

        "no_z3_progenitor_fraction": no_prog,
    })


factor_summary = (
    factor_draws
    .groupby(
        ["source_id", "sfh_model"],
        sort=False,
    )
    .apply(summarize_group)
    .reset_index()
)

factor_summary = factor_summary.merge(
    pilot[
        [
            "source_id",
            "catalog_z",
            "logM",
            "g_E1J",
        ]
    ],
    on="source_id",
    how="left",
    validate="many_to_one",
)

factor_summary["g50_over_E1J"] = (
    factor_summary["g_p50"]
    / factor_summary["g_E1J"]
)

display(
    factor_summary.sort_values(
        ["source_id", "sfh_model"]
    )
)

factor_summary.to_csv(
    RESULT_DIR / "SFH_model_gbag_summary.csv",
    index=False,
)



## 12. Cross-model robustness table

This does **not** force a pass/fail threshold. It reports how much the median \(g_{\rm BAG}\) changes when only the SFH family is changed.

If the model-to-model median span is large, or the individual posterior intervals span orders of magnitude, the backward luminosity correction is SFH-prior/model dependent even if every model fits the observed SED well.


In [ ]:

robust_rows = []

for oid in TEST_IDS:

    sub = factor_summary.loc[
        factor_summary["source_id"] == oid
    ].copy()

    positive_medians = sub.loc[
        np.isfinite(sub["g_p50"])
        & (sub["g_p50"] > 0),
        "g_p50",
    ].to_numpy(dtype=float)

    if len(positive_medians) >= 2:
        median_span_factor = (
            np.nanmax(positive_medians)
            / np.nanmin(positive_medians)
        )
        median_span_dex = (
            np.log10(np.nanmax(positive_medians))
            - np.log10(np.nanmin(positive_medians))
        )
    else:
        median_span_factor = np.nan
        median_span_dex = np.nan

    robust_rows.append({
        "source_id": oid,
        "n_models": len(sub),
        "min_model_g50": (
            np.nanmin(positive_medians)
            if len(positive_medians) else np.nan
        ),
        "max_model_g50": (
            np.nanmax(positive_medians)
            if len(positive_medians) else np.nan
        ),
        "model_median_span_factor": median_span_factor,
        "model_median_span_dex": median_span_dex,
        "min_P_g_gt_1": np.nanmin(sub["P_g_gt_1"]),
        "max_P_g_gt_1": np.nanmax(sub["P_g_gt_1"]),
        "min_formed_fraction_p50": np.nanmin(
            sub["formed_fraction_p50"]
        ),
        "max_formed_fraction_p50": np.nanmax(
            sub["formed_fraction_p50"]
        ),
    })


robustness = pd.DataFrame(robust_rows)

display(robustness)

robustness.to_csv(
    RESULT_DIR / "SFH_model_robustness_summary.csv",
    index=False,
)



## 13. Compact comparison plot

This plot shows the median and 16-84% interval of \(g_{\rm BAG}\) for each SFH family. The y-axis is logarithmic because the posterior can span orders of magnitude.


In [ ]:

for oid in TEST_IDS:

    sub = (
        factor_summary.loc[
            factor_summary["source_id"] == oid
        ]
        .set_index("sfh_model")
        .loc[["dblplaw", "delayed", "continuity"]]
        .reset_index()
    )

    x = np.arange(len(sub))
    y = np.asarray(sub["g_p50"], dtype=float)

    low = y - np.asarray(sub["g_p16"], dtype=float)
    high = np.asarray(sub["g_p84"], dtype=float) - y

    # Avoid invalid log plotting for exact zeros.
    yplot = np.where(y > 0, y, np.nan)

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.errorbar(
        x,
        yplot,
        yerr=np.vstack([low, high]),
        fmt="o",
        capsize=4,
    )

    ax.axhline(
        float(sub["g_E1J"].iloc[0]),
        linestyle="--",
        label="E1J",
    )

    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(sub["sfh_model"])
    ax.set_ylabel(r"$g_{\rm BAG}$")
    ax.set_title(f"ID {oid}: SFH-model sensitivity")
    ax.legend()

    fig.tight_layout()

    fig.savefig(
        RESULT_DIR / f"{oid}_SFH_model_gbag_comparison.png",
        dpi=180,
        bbox_inches="tight",
    )

    plt.show()



## 14. What to send back

When the notebook finishes, send these three CSV files:

1. `SFH_model_fit_quality.csv`
2. `SFH_model_gbag_summary.csv`
3. `SFH_model_robustness_summary.csv`

They are written to:

`/home/bahareh/Desktop/Projects/Passive_Spiral/Data/bagpipes_GOLD91_pilot_v1/sfh_model_robustness/`

Do **not** run BAGPIPES for the other 89 GOLD91 galaxies yet. The purpose of this notebook is to decide whether the backward luminosity factor is identifiable enough to justify a larger EBAG branch.
